In [1]:
import os
import PIL
from PIL import Image
import pandas as pd
import numpy as np

In [2]:
DATA_PATH = '../../compressed_dataset'
list_rows = []
for folder in os.listdir(DATA_PATH):
    for file in os.listdir(os.path.join(DATA_PATH,folder)):
        with PIL.Image.open(os.path.join(DATA_PATH,folder,file)) as img:
            width, hight = img.size
            form = img.format
            temp_dict={
                'label' : folder,
                'width' : width,
                'hight' : hight,
                'format' : form,
                'path' : os.path.join(DATA_PATH,folder,file)
            }
            list_rows.append(temp_dict)
df = pd.DataFrame(list_rows)
countries = df.label.unique().tolist()

In [16]:
import torch
import clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

image = preprocess(Image.open(f"{DATA_PATH}/Germany/canvas_1629281464.jpg")).unsqueeze(0).to(device)
text = clip.tokenize(countries).to(device)
#text = clip.tokenize(["Brazil", "Germany", "United States", "Curacao"]).to(device)

with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    
    logits_per_image, logits_per_text = model(image, text)
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()

#print(text)
#print("Label probs:", probs)  # prints: [[0.9927937  0.00421068 0.00299572]]
max_index = probs.argmax()
print(probs[0][max_index], countries[max_index])

0.09792508 Germany


In [20]:
df_100 = df.sample(n=100, replace=False)
print(df_100)

                label  width  hight format  \
15090   United States   1536    662   JPEG   
47771           Italy   1536    662   JPEG   
20732          Canada   1536    662   JPEG   
11324   United States   1536    662   JPEG   
387       Netherlands   1536    662   JPEG   
...               ...    ...    ...    ...   
23315  United Kingdom   1536    662   JPEG   
31262        Slovenia   1536    662   JPEG   
17389          France   1536    662   JPEG   
28303        Thailand   1536    662   JPEG   
22061  United Kingdom   1536    662   JPEG   

                                                    path  
15090  ../../compressed_dataset/United States/canvas_...  
47771  ../../compressed_dataset/Italy/canvas_16295049...  
20732  ../../compressed_dataset/Canada/canvas_1629867...  
11324  ../../compressed_dataset/United States/canvas_...  
387    ../../compressed_dataset/Netherlands/canvas_16...  
...                                                  ...  
23315  ../../compressed_dataset/Un

In [21]:
import torch
import clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

predicted_values = []
reference_values = []
text = clip.tokenize(countries).to(device)

for index, row in df_100.iterrows():
    reference_values.append(row['label'])
    image = preprocess(Image.open(row['path'])).unsqueeze(0).to(device)
    
    with torch.no_grad():
        
        image_features = model.encode_image(image)
        text_features = model.encode_text(text)
    
        logits_per_image, logits_per_text = model(image, text)
        probs = logits_per_image.softmax(dim=-1).cpu().numpy()
        
    max_index = probs.argmax()
    predicted_values.append(countries[max_index])
    
print(reference_values)
print(predicted_values)

['United States', 'Italy', 'Canada', 'United States', 'Netherlands', 'Hong Kong', 'United States', 'South Korea', 'Thailand', 'Ukraine', 'United States', 'Thailand', 'Ecuador', 'United States', 'Russia', 'Japan', 'Germany', 'Romania', 'South Africa', 'Sweden', 'Japan', 'United States', 'United States', 'India', 'United Kingdom', 'Indonesia', 'United States', 'Colombia', 'United States', 'Thailand', 'Australia', 'Spain', 'France', 'Eswatini', 'United States', 'Ghana', 'United States', 'United Kingdom', 'Japan', 'Finland', 'Canada', 'Spain', 'United States', 'Austria', 'Russia', 'United States', 'Russia', 'Mongolia', 'United States', 'South Africa', 'United States', 'France', 'Canada', 'United States', 'Spain', 'United States', 'Norway', 'United States', 'Uganda', 'France', 'Canada', 'Netherlands', 'Germany', 'Mexico', 'Albania', 'Singapore', 'France', 'Singapore', 'Romania', 'United States', 'Mexico', 'Finland', 'Canada', 'United States', 'Norway', 'Australia', 'United States', 'Canada'

In [25]:
total_n = len(reference_values)
correct = 0
for i in range(total_n):
    if reference_values[i] == predicted_values[i]:
        correct += 1
        
print(f"Total image number: {total_n}")
print(f"Correct classified images: {correct}")
print(f"Accuracy: {correct/total_n}")
print(f"Number of possible countries: {len(countries)}")


Total image number: 100
Correct classified images: 32
Accuracy: 0.32
Number of possible countries: 124
